$$\large{\color{yellow}{\text{MDS6304 Deep Learning ETEPW}}}$$

$$\large\text{Theme}: \underline{\text{building a neural network from scratch using efficient computations in PyTorch}}$$

---

Load libraries

---

In [ ]:
## Load libraries
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import tensorflow as tf
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
plt.style.use('dark_background')
%matplotlib inline

---

Mount Google drive

---

In [ ]:
## Mount Google drive folder if running in Colab
if('google.colab' in sys.modules):
    from google.colab import drive
    drive.mount('/content/drive', force_remount = True)
    DIR = '/content/drive/MyDrive/Colab Notebooks/MAHE/Office of Online Education/MDS6304_Webinar_October2025'
    DATA_DIR = DIR + '/Data/'
else:
    DATA_DIR = 'Data/'

---

Load diabetes data (binary & multilabel classification)

---

In [ ]:
## Load diabetes data
file = DATA_DIR+'diabetes.csv'
df= pd.read_csv(file, header = 0).dropna()

## Train and test split of the data
X = df.loc[:, df.columns != 'Outcome']
y = df['Outcome']
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

# Standardize data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Convert train and test data to torch tensors (note that Y should be a 1-column matrix)
X_train = torch.tensor(X_train, dtype = torch.float64)
X_test = torch.tensor(X_test, dtype = torch.float64)
# Label encode class labels (do the following if using logistic regression for multilabel classification)
Y_train = torch.tensor(Y_train.values).reshape(-1, 1)
Y_test = torch.tensor(Y_test.values).reshape(-1, 1)
# One-hot encode class labels (do the following if using softmax for multilabel classification)
#Y_train =  nn.functional.one_hot(torch.tensor(Y_train.values, dtype = torch.int64))
#Y_test = nn.functional.one_hot(torch.tensor(Y_test.values, dtype = torch.int64))


num_samples = X_train.shape[0]
num_features = X_train.shape[1]
num_labels = len(np.unique(Y_train))

print('Diabetes data set')
print('---------------------')
print('Number of training samples = %d'%(num_samples))
print('Number of features = %d'%(num_features))

---

Load MNIST Data (multilabel classification)

---

In [ ]:
## Load MNIST data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()
X_train = torch.tensor(X_train.reshape(X_train.shape[0], X_train.shape[1]*X_train.shape[2]))
X_test = torch.tensor(X_test.reshape(X_test.shape[0], X_test.shape[1]*X_test.shape[2]))

num_samples = X_train.shape[0]
num_labels = len(np.unique(y_train))
num_features = X_train.shape[1]

# One-hot encode class labels
Y_train =  nn.functional.one_hot(torch.tensor(y_train, dtype = torch.int64))
Y_test = nn.functional.one_hot(torch.tensor(y_test, dtype = torch.int64))

# Normalize the samples (images) using the training data
xmax = torch.amax(X_train) # 255
xmin = torch.amin(X_train) # 0
X_train = (X_train - xmin) / (xmax - xmin) # all train features turn into a number between 0 and 1
X_test = (X_test - xmin) / (xmax - xmin)

print('MNIST set')
print('---------------------')
print('Number of training samples = %d'%(num_samples))
print('Number of features = %d'%(num_features))
print('Number of output labels = %d'%(num_labels))

---

Load housing data (regression)

---

In [ ]:
## Load housing data
file = DATA_DIR+'houseprices_cleaned.csv'
df= pd.read_csv(file, header = 0).dropna()

## Train and test split of the data
X = df[['area', 'rent']]
y = df['price_per_sqft']
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

# Standardize data
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Convert train and test data to numpy arrays (note that Y should be a 1-column matrix)
X_train = torch.tensor(X_train, dtype = torch.float32)
X_test = torch.tensor(X_test, dtype = torch.float32)
Y_train = torch.tensor(Y_train.values, dtype = torch.float32).reshape(-1, 1)
Y_test = torch.tensor(Y_test.values, dtype = torch.float32).reshape(-1, 1)

num_samples = X_train.shape[0]
num_features = X_train.shape[1]

print('Housing data set')
print('---------------------')
print('Number of training samples = %d'%(num_samples))
print('Number of features = %d'%(num_features))

---

A generic layer class with forward and backward methods

----

In [ ]:
class Layer(nn.Module):
  def __init__(self):
    super(Layer, self).__init__()
    self.input = None
    self.output = None

  def forward(self, input):
    raise NotImplementedError("Forward propagation not implemented")

  def backward(self, output_gradient, learning_rate):
    raise NotImplementedError("Backward propagation not implemented")

---

Binary crossentropy (BCE) loss and its gradient for the batch samples (for binary classification)

---

In [ ]:
## Define the loss function and its gradient
class BinaryCrossEntropyLoss:
  def forward(self, Y, Yhat):
    epsilon = torch.tensor(1.0e-08)
    return torch.mean(-Y*torch.log(Yhat+epsilon)-(1-Y)*torch.log(1-Yhat+epsilon))

  def backward(self, Y, Yhat):
    return (Yhat-Y)/(Yhat*(1-Yhat))

---

Categorical crossentropy (CCE) loss and its gradient for the batch samples (for multilabel classification)

---

In [ ]:
## Define the loss function and its gradient
class CategoricalCrossEntropyLoss:
  def forward(self, Y, Yhat):
    return torch.mean(-torch.log(torch.sum(Y * Yhat, dim = 1)))

  def backward(self, Y, Yhat):
    return -Y/Yhat

---

Mean squared error (MSE) loss and its gradient for the batch samples (for regression)

---

In [ ]:
## Define the loss function and its gradien
class MeanSquaredErrorLoss:
  def forward(self, Y, Yhat):
    return 0.5*torch.mean((Y-Yhat)**2)

  def backward(self, Y, Yhat):
    return Yhat-Y

---

ReLU activation layer class

---

In [ ]:
class ReLU(Layer):
  def forward(self, input):
    self.input = input
    self.output = self.input * (self.input > 0.0)
    return self.output

  def backward(self, output_gradient, learning_rate = None):
    relu_local_gradient = 1.0 * (self.input > 0.0)
    if output_gradient.shape[1] > 1:
      return output_gradient[:, :-1] * relu_local_gradient
    else:
      return output_gradient * relu_local_gradient

---

Sigmoid activation layer class

---

In [ ]:
class Sigmoid(Layer):
  def forward(self, input):
    self.input = input
    self.output = 1.0 / (1.0 + torch.exp(-self.input))
    return self.output

  def backward(self, output_gradient, learning_rate = None):
    sigmoid_local_gradient = self.output * (1 - self.output)
    if output_gradient.shape[1] > 1:
      return output_gradient[:, :-1] * sigmoid_local_gradient
    else:
      return output_gradient * sigmoid_local_gradient

---

Softmax activation layer class


---

In [ ]:
## Softmax activation layer class
class Softmax(Layer):
  def forward(self, input):
    self.input = input
    exp_values = torch.exp(self.input - torch.max(self.input, dim = 1, keepdim = True).values)
    self.output = exp_values / torch.sum(exp_values, dim = 1, keepdim = True)
    return self.output

  def backward(self, output_gradient, learning_rate = None):
    I = torch.eye(self.output.shape[1])
    softmax_local_gradient = (I - self.output[:, :, None]) * self.output[:, None, :]
    # Calculate gradient flowing back on the input side of the softmax layer
    input_gradient = torch.einsum('ij,ijk->ik', output_gradient, softmax_local_gradient)
    return input_gradient

---

Dense layer class

---

In [ ]:
## Dense layer class
class Dense(Layer):
  def __init__(self, input_size, output_size, reg_strength = 0.0):
    super(Dense, self).__init__()
    self.weights = nn.Parameter(torch.randn(input_size+1, output_size) * 0.01)
    with torch.no_grad():
      self.weights[-1].fill_(0.01)  # set all bias values to the same nonzero constant
    self.reg_strength = reg_strength
    self.reg_loss = None

  def forward(self, input):
    self.input = torch.hstack([input, torch.ones(input.shape[0], 1)]) # bias trick
    self.output= torch.matmul(self.input, self.weights)
    # Calculate regularization loss
    self.reg_loss = self.reg_strength * torch.sum(self.weights[:-1, :] * self.weights[:-1, :])
    return self.output, self.reg_loss

  def backward(self, output_gradient, learning_rate):
    # Calculate local gradient w.r.t. weights
    dense_local_weights_gradient = (1/output_gradient.shape[0]) * (torch.einsum('ij,ik->jk', self.input, output_gradient))
    # Add the regularization gradient here
    dense_local_weights_gradient += 2 * self.reg_strength * torch.vstack([self.weights[:-1, :], torch.zeros(1, self.weights.shape[1])])
    # Calculate gradient flowing back on the input side of the dense layer
    input_gradient = torch.matmul(output_gradient, self.weights.T)
    # Update weights for dense layer
    with torch.no_grad():
      self.weights.data += learning_rate * (-dense_local_weights_gradient)
    # Return gradient flowing back on the input side of the dense layer
    return(input_gradient)

---

Neural network class for:

 1. Binary classification (loss function is binary crossentropy and last layer has one node that is sigmoid-activated).
 2. Multilabel classification (loss function is categorical crossentropy and last layer is softmax-activated).
 3. Regression (loss function is MSE loss and last layer is dense with one node).

---

In [ ]:
class NeuralNetwork:
  def __init__(self, num_features, num_labels, loss_fn,
               learning_rate, reg_strength = 0.0):
    self.num_features = num_features
    self.num_labels = num_labels
    self.loss_fn = loss_fn
    self.learning_rate = learning_rate
    self.reg_strength = reg_strength
    self.reg_loss = None
    # Architecture
    self.layers = [
        Dense(self.num_features, 512, self.reg_strength),
        ReLU(),
        Dense(512, num_labels, self.reg_strength),
        Softmax()
        #Sigmoid
        ]

  # Forward propagation
  def forward(self, x):
    self.reg_loss = 0.0
    for layer in self.layers:
      if hasattr(layer, 'weights'):
        x, reg_loss = layer.forward(x)
        self.reg_loss += reg_loss
      else:
        x = layer.forward(x)
    return x

  # Backward propagation
  def backward(self, loss_gradient):
    for layer in reversed(self.layers):
      loss_gradient = layer.backward(loss_gradient, self.learning_rate)

---

Train a neural network for:

 1. Binary classification (loss function is binary crossentropy).
 2. Multilabel classification (loss function is categorical crossentropy).
 3. Regression (loss function is MSE).

---

In [ ]:
# Initialize model and optimizer
learning_rate = 1e-03
epochs = 100
reg_strength = 0.1
batch_size = 32

# Choose appropriate loss function
#loss_fn = BinaryCrossEntropyLoss()
loss_fn = CategoricalCrossEntropyLoss()
#loss_fn = MeanSquaredErrorLoss()

# Data loader for batch processing
train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size = batch_size, shuffle = True)
test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size = X_test.shape[0], shuffle = False)

model = NeuralNetwork(num_features, num_labels, loss_fn,
                              learning_rate, reg_strength)

# Create empty list to store training and test losses over each epoch
train_loss = [None]*epochs
test_loss = [None]*epochs

for epoch in range(epochs):
  loss = 0.0

  for x_batch, y_batch in train_loader:
    # Forward pass
    predictions = model.forward(x_batch)
    loss = loss + model.loss_fn.forward(y_batch, predictions) + model.reg_loss

    # Backward pass
    loss_gradient = model.loss_fn.backward(y_batch, predictions)
    model.backward(loss_gradient)

  train_loss[epoch] = loss.detach().numpy() / len(train_loader)

  # Test loss calculation
  loss = 0.0

  with torch.no_grad():
    for x_batch, y_batch in test_loader:
      predictions = model.forward(x_batch)
      loss = loss + model.loss_fn.forward(y_batch, predictions) + model.reg_loss

  test_loss[epoch] = loss.detach().numpy()

  print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss[epoch]:.4f}, Test Loss: {test_loss[epoch]:.4f}")

---

Plot training and test loss vs. epoch

---

In [ ]:
## Plot train and test loss as a function of epoch:
fig, ax = plt.subplots(1, 1, figsize = (4, 4))
fig.tight_layout(pad = 4.0)
ax.plot(train_loss, 'b', label = 'Train')
ax.plot(test_loss, 'r', label = 'Test')
ax.set_xlabel('Epoch', fontsize = 12)
ax.set_ylabel('Loss value', fontsize = 12)
ax.legend()
ax.set_title('Loss vs. Epoch', fontsize = 14);

---

Assess model performance on test data

---

In [ ]:
## Assess model performance on test data (multilabel classification)
with torch.no_grad():
  for x_batch, y_batch in test_loader:
    predictions = model.forward(x_batch)

# Predicted labels for binary classification
#ypred = (predictions >= 0.45)
# Predicted labels for multilabel classification
ypred = torch.argmax(predictions, dim = 1).float()
# Predicted values for regression
ypred = predictions

# True labels for binary classification
#ytrue = Y_test
# True labels for multilabel classification
ytrue = np.argmax(Y_test, axis = 1).float()
# True labels for regression
ytrue = Y_test

# Classifier performace
#print(f'Accuracy on test data = {(torch.mean((ytrue == ypred).float())*100):3.2f}')
# Print confusion matrix
#print(confusion_matrix(ytrue, ypred))

# Regressor performance
print(f'Mean-squared error on test data = {torch.mean((ytrue-ypred)**2):0.2f}')
# Plot true and predicted values
fig, ax = plt.subplots(1, 1, figsize = (4, 4))
fig.tight_layout(pad = 4.0)
ax.scatter(ytrue.detach().numpy().flatten(), ypred.detach().numpy().flatten(), s = 5, color = 'r')
ax.plot(ax.get_xlim(), ax.get_ylim(), ls = "--", c = ".3")
ax.set_xlabel('True values', fontsize = 12)
ax.set_ylabel('Predicted values', fontsize = 12)
ax.set_title('True vs. Predicted Values', fontsize = 14);